In [15]:
# --- AEWS ENVIRONMENT SETUP ---
import os, sys

PROJECT_ROOT = r"E:\Pranav\Engineering\Projects\AEWS"
os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("✔ Working directory:", os.getcwd())

✔ Working directory: E:\Pranav\Engineering\Projects\AEWS


In [16]:
import pandas as pd

isi_df = pd.read_csv("data/processed/isi_scores.csv")
lifecycle_df = pd.read_csv("data/processed/lifecycle_clusters.csv")

print("ISI:", isi_df.shape)
print("Lifecycle:", lifecycle_df.shape)

isi_df.head()


ISI: (12004, 10)
Lifecycle: (12004, 11)


,state,district,year_month,enrol_activity,demo_activity,bio_activity,enrol_norm,demo_norm,bio_norm,isi_score
0,100000,100000,2025-02,100003,200002,14139195,6.364787e-10,0.000019,0.003578,0.001439
1,100000,100000,2025-03,100001,11161793,12649948,0.000000e+00,0.002297,0.003198,0.002198
2,100000,100000,2025-08,100001,9673377,9673415,0.000000e+00,0.001988,0.002439,0.001771
3,100000,100000,2025-09,100001,13393929,7441171,0.000000e+00,0.002761,0.001870,0.001852
4,100000,100000,2025-11,100002,14882372,12650221,3.182393e-10,0.003070,0.003198,0.002507


In [17]:
df = isi_df.merge(
    lifecycle_df[["state", "district", "year_month", "lifecycle_cluster"]],
    on=["state", "district", "year_month"],
    how="left"
)

df = df.sort_values(["state", "district", "year_month"])

df.head()


,state,district,year_month,enrol_activity,demo_activity,bio_activity,enrol_norm,demo_norm,bio_norm,isi_score,lifecycle_cluster
0,100000,100000,2025-02,100003,200002,14139195,6.364787e-10,0.000019,0.003578,0.001439,0
1,100000,100000,2025-03,100001,11161793,12649948,0.000000e+00,0.002297,0.003198,0.002198,0
2,100000,100000,2025-08,100001,9673377,9673415,0.000000e+00,0.001988,0.002439,0.001771,0
3,100000,100000,2025-09,100001,13393929,7441171,0.000000e+00,0.002761,0.001870,0.001852,0
4,100000,100000,2025-11,100002,14882372,12650221,3.182393e-10,0.003070,0.003198,0.002507,0


In [18]:
# --- Step 6.1: Define risk using ISI percentiles ---

low_cut = df["isi_score"].quantile(0.70)
high_cut = df["isi_score"].quantile(0.90)

print("Low / Medium:", low_cut)
print("Medium / High:", high_cut)

def label_risk(isi):
    if isi <= low_cut:
        return 0   # 🟢 Low
    elif isi <= high_cut:
        return 1   # 🟡 Medium
    else:
        return 2   # 🔴 High

df["risk_label"] = df["isi_score"].apply(label_risk)

df["risk_label"].value_counts()


Low / Medium: 0.00908549133723356
Medium / High: 0.026395678413029294


risk_label
0    8403
1    2400
2    1201
Name: count, dtype: int64

In [19]:
# --- Step 6.2: Shift risk to NEXT month (Early Warning logic) ---

df["risk_label_next"] = (
    df
    .groupby(["state", "district"])["risk_label"]
    .shift(-1)
)

# Remove rows where next month is unknown
df = df.dropna(subset=["risk_label_next"])

df["risk_label_next"] = df["risk_label_next"].astype(int)

df[["year_month", "risk_label", "risk_label_next"]].head(10)


,year_month,risk_label,risk_label_next
0,2025-02,0,0
1,2025-03,0,0
2,2025-08,0,0
3,2025-09,0,0
4,2025-11,0,0
5,2025-12,0,0
7,2025-01,0,0
8,2025-02,0,0
9,2025-04,0,0
10,2025-09,0,0


In [20]:
feature_cols = [
    "enrol_norm",
    "demo_norm",
    "bio_norm",
    "lifecycle_cluster"
]

X = df[feature_cols]
y = df["risk_label_next"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

y.value_counts(normalize=True)


Features shape: (10959, 4)
Target shape: (10959,)


risk_label_next
0    0.701433
1    0.201661
2    0.096907
Name: proportion, dtype: float64

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [22]:
from src.models.risk_classifier import RiskClassifier

classifier = RiskClassifier()

classifier.train(X_train, y_train)


In [23]:
classifier.evaluate(X_test, y_test)


              precision    recall  f1-score   support

           0       0.88      0.85      0.87      1538
           1       0.55      0.69      0.61       442
           2       0.38      0.26      0.31       212

    accuracy                           0.76      2192
   macro avg       0.60      0.60      0.59      2192
weighted avg       0.76      0.76      0.76      2192



In [24]:
df["predicted_risk_next"] = classifier.predict(X)

df[[
    "state",
    "district",
    "year_month",
    "risk_label_next",
    "predicted_risk_next"
]].head(10)
  

,state,district,year_month,risk_label_next,predicted_risk_next
0,100000,100000,2025-02,0,0
1,100000,100000,2025-03,0,0
2,100000,100000,2025-08,0,0
3,100000,100000,2025-09,0,0
4,100000,100000,2025-11,0,0
5,100000,100000,2025-12,0,0
7,Andaman & Nicobar Islands,Andamans,2025-01,0,0
8,Andaman & Nicobar Islands,Andamans,2025-02,0,0
9,Andaman & Nicobar Islands,Andamans,2025-04,0,0
10,Andaman & Nicobar Islands,Andamans,2025-09,0,0


In [25]:
df[[
    "state",
    "district",
    "year_month",
    "predicted_risk_next"
]].to_csv(
    "outputs/predictions/aews_risk_signals.csv",
    index=False
)

print("✔ AEWS next-month risk signals saved")


✔ AEWS next-month risk signals saved
